# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Prediction point: when the warranty is sold. All columns in `columns.csv` are known at sale time (age at sale, price, channel, daily use reported at registration), so all are eligible; `device_serial` is an identifier and is dropped.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.device_serial.nunique(), 'unique serials')
print(df.claim_within_year.mean())

## 1. Test set
There is no date column, so no temporal ordering is available, and every `device_serial` appears exactly once, so there are no repeated entities to group. A random stratified 20% hold-out is therefore appropriate. It is not touched until the final step.
Note: `daily_hours` has values above 24, which is impossible as a daily average; they are kept as-is (self-reported data) and flagged here.

In [ ]:
print('rows with daily_hours>24:', (df.daily_hours>24).sum())
assert df.device_serial.is_unique
FEATURES=['channel','device_age_years','price_eur','daily_hours']
X=df[FEATURES]; y=df.claim_within_year
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te),y_tr.mean(),y_te.mean())

## 2-4. Candidates and CV scheme (declared before any scoring)
Candidates:
1. `baseline`: predicts the training prior (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression on scaled numeric + one-hot channel.
3. `hgb_shallow`, `hgb_deep`: gradient boosting (max_depth 2 / max_depth 4, lr 0.05, 150 iterations).

Selection metric: mean cross-validated ROC AUC (also reporting PR AUC given about 12.6% prevalence). Cross-validation: 5-fold stratified, inside the training rows only (no entities repeat, no time order). All preprocessing sits inside the Pipeline so it is fitted per fold. The decision threshold is chosen on out-of-fold predictions of the winner, maximising F1.

In [ ]:
num=['device_age_years','price_eur','daily_hours']
def prep(scale):
    return ColumnTransformer([('num',StandardScaler() if scale else 'passthrough',num),
                              ('cat',OneHotEncoder(handle_unknown='ignore'),['channel'])])
def hgb(d): return HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=150,random_state=SEED)
CANDIDATES={
 'baseline':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'logreg_C10':Pipeline([('p',prep(True)),('m',LogisticRegression(C=10,max_iter=1000))]),
 'hgb_shallow':Pipeline([('p',prep(False)),('m',hgb(2))]),
 'hgb_deep':Pipeline([('p',prep(False)),('m',hgb(4))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)

## 5. Model selection on cross-validation only

In [ ]:
oof={}; rows=[]
for name,pipe in CANDIDATES.items():
    p=cross_val_predict(pipe,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
    oof[name]=p
    fold_auc=[roc_auc_score(y_tr.iloc[te],p[te]) for _,te in cv.split(X_tr,y_tr)]
    rows.append((name,np.mean(fold_auc),np.std(fold_auc),average_precision_score(y_tr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd','cv_pr_auc']).sort_values('cv_auc_mean',ascending=False)
print(cvres.to_string(index=False))

In [ ]:
best=cvres.iloc[0].model
ths=np.linspace(0.02,0.6,59)
f1s=[f1_score(y_tr,oof[best]>=t) for t in ths]
THRESH=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(THRESH,3),'| CV F1:',round(max(f1s),3))
print('CV F1 of always-positive:',round(f1_score(y_tr,np.ones(len(y_tr))),3))

**Decision (recorded before test scoring):** the model and threshold above, selected on cross-validation alone. The test set has not been used yet.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=CANDIDATES[best].fit(X_tr,y_tr)
p=final.predict_proba(X_te)[:,1]; pred=p>=THRESH
rng=np.random.default_rng(SEED); aucs=[]
for _ in range(1000):
    i=rng.integers(0,len(y_te),len(y_te))
    if y_te.iloc[i].nunique()>1: aucs.append(roc_auc_score(y_te.iloc[i],p[i]))
print(f'TEST ROC AUC {roc_auc_score(y_te,p):.3f} (95% bootstrap CI {np.percentile(aucs,2.5):.3f}-{np.percentile(aucs,97.5):.3f})')
print(f'TEST PR AUC {average_precision_score(y_te,p):.3f} (prevalence {y_te.mean():.3f})')
print(f'at threshold {THRESH:.2f}: precision {precision_score(y_te,pred):.3f} recall {recall_score(y_te,pred):.3f} F1 {f1_score(y_te,pred):.3f}')

## Conclusion
See the printed test results above: ROC AUC with its bootstrap interval, versus 0.5 for the trivial baseline, and PR AUC versus the 12.6% prevalence. These are the headline numbers; the test set was scored once.